# Run a PCR plate with ABI 7500 Fast Dx

```{device-card} thermo-fisher-abi7500-fast-dx
```

This notebook runs a finite thermal program from Ubuntu, collects original camera frames during
selected holds, and releases the plate after cooling. It uses the public PLR API and a persistent
journal. Windows and vendor DLLs are not required.

```{warning}
Experimental integration. A water-plate engineering run does not validate amplification, current
calibration or quantitative assay performance. Check the [hardware report](hardware-validation.md)
for the tested camera, temperatures and number of cycles. Every hardware session needs explicit
approval. The heated cover can remain near 105 °C after the sample cools.
```

The example is Standard mode, 20 µL: 95 °C/120 s, five repetitions of 95 °C/15 s and
60 °C/60 s, then 30 °C/60 s. It collects physical filter 5 at 30/10/5 ms during each 60 °C
hold. This is an acquisition/timing test on water, not a reagent protocol. The five-cycle example
produces 15 frames. A forty-cycle assay is a separate session requiring suitable chemistry,
exposures, unit-specific well mapping/calibration and a longer observation budget.

Install PLR with USB support and follow the [connection guide](hello-world.ipynb). Keep the live
journal on local storage outside synchronized folders; copy completed exports for sharing.

## Imports — offline

In [ ]:
import asyncio
import hashlib
import json
import math
from pathlib import Path

from pylabrobot.thermo_fisher import ABI7500FastDx
from pylabrobot.thermo_fisher.abi7500_fast_dx import (
  CollectionFilter,
  CollectionPlan,
  ThermalCycle,
  ThermalProgram,
  ThermalStep,
)

## Describe the method — offline

The captured metadata below selects the supported Standard/20 µL/block-2/tube-0 configuration.
PLR replaces its method/user labels with the run ID and `PLR`. These are native metadata, not
an EDS file. The final cooling hold is part of the uploaded program, with no optical collection.
Temperature and exposure units are °C and seconds. The 45-second collection window includes
filter motion, all three images and homing; it never extends the thermal hold.

For a separate forty-cycle run, change `repetitions` to 40 and budget at least the planned holds,
ramps, cover warmup and cleanup. The saved vendor assay lasted about 89 minutes; 7,200 seconds
is a starting observation budget, not a worst-case bound. The 30/10/5 ms exposures are engineering
settings and must be chosen for the reaction plate before running an assay.

In [ ]:
def build_pcr_program(repetitions: int = 5) -> tuple[ThermalProgram, CollectionPlan]:
  """Build the finite Standard/20 µL engineering method without opening USB."""
  metadata = bytes.fromhex(
    "4d736d616c6c00000000000000000000004d617961000000630a02020a0b0c0000620a02020a030100001400020000"
  )
  program = ThermalProgram(
    metadata,
    (
      ThermalCycle(1, (ThermalStep(95, 120),)),
      ThermalCycle(repetitions, (ThermalStep(95, 15), ThermalStep(60, 60))),
      ThermalCycle(1, (ThermalStep(30, 60),)),
    ),
  )
  plan = CollectionPlan(
    steps=((1, 1),),
    filters=(CollectionFilter(5, (0.030, 0.010, 0.005)),),
    read_window=45,
    park_between_collections=True,
    home_after_collections=tuple(range(1, repetitions + 1)),
  )
  plan.points(program)
  return program, plan

## Select the instrument and run — offline

Enter this instrument's USB serial and full camera description from its previous identity record.
Use its existing persistent journal. Never change journal paths to retry a run, and never reuse
another instrument's calibration. A run ID denotes one intentional experiment, even after a
connection failure. Keep the same ID when inspecting an interrupted run; do not automatically
restart it with a new ID.

The output folder is reserved before USB is opened. Rerunning the execution cell fails if that
folder already exists. The notebook starts with hardware execution disabled.

In [ ]:
usb_serial = ""  # USB serial, not the serial printed on the thermal cycler.
expected_camera = ""  # Full camera identity associated with this physical instrument.
run_id = "pcr5-001"  # At most 16 ASCII letters, digits, underscores or hyphens.
journal_path = (
  Path.home() / ".local/state/pylabrobot/abi7500" / f"usb-{usb_serial}" / "runs.sqlite3"
)
output = Path.home() / "Documents/abi7500-results" / run_id
program, collection_plan = build_pcr_program(repetitions=5)
observation_timeout = 1800.0
approved_session = False
print(
  f"{len(collection_plan.points(program))} collections, "
  f"{3 * len(collection_plan.points(program))} images; output: {output}"
)

## Define completion and failure cleanup — offline

Cleanup reconciles the same run and cancels only a positively identified active run. It restores
parked, dark optics after inactivity, and releases only a normally completed run with matching
history and sample/block temperatures at most 40 °C. A failure at high temperature leaves the
plate loaded. It never submits another program, retries an exposure or retries release.
An unresolved communication or release state requires inspection using the same journal.
Closing USB alone neither cancels a run nor cools the heated cover.

In [ ]:
async def finish_pcr(
  instrument: ABI7500FastDx, run_id: str, expected_camera: str, start_attempted: bool
) -> dict:
  """Reconcile this session, restore idle optics, and release a completed cooled plate."""
  if not instrument.is_connected:
    await instrument.stop()
    await instrument.setup()
  identity = await instrument.get_identity()
  if identity.camera_description != expected_camera or identity.thermal_version != "G2.10":
    raise RuntimeError("Identity differs during cleanup; no actuation")
  run = None
  if start_attempted:
    deadline = asyncio.get_running_loop().time() + 30
    while True:
      run = await instrument.get_run(run_id)
      if run.state in ("running", "cancelling"):
        run = await instrument.cancel_run(run_id)
      if run.is_terminal:
        break
      if asyncio.get_running_loop().time() >= deadline:
        raise RuntimeError("Run ownership unresolved; inspect the same journal")
      await asyncio.sleep(0.5)
    if run.plate_release_state == "uncertain":
      raise RuntimeError("Release state is uncertain; do not repeat motion")
  status = await instrument.get_status()
  if status.phase != "inactive" or status.state_code not in (1, 2):
    raise RuntimeError("Instrument is not idle for optical cleanup")
  await instrument.set_filter_position(0)
  optics = await instrument.set_lamp(False)
  summary = {"optics": optics.to_dict(), "plate_released": False}
  if run is not None:
    summary["run"] = run.to_dict()
    if run.state == "completed":
      status = await instrument.get_status()
      if max(status.sample_temperature, status.block_temperature) > 40:
        raise RuntimeError("Completed run is still hot; plate retained")
      plate = await instrument.release_plate(run_id)
      summary["plate_released"] = plate.state == "unloaded"
      summary["run"] = (await instrument.get_run(run_id)).to_dict()
  status = await instrument.get_status()
  summary.update(
    sample_temperature=status.sample_temperature,
    block_temperature=status.block_temperature,
    cover_temperature=status.cover_temperature,
  )
  return summary

## Define the run workflow — offline

The collector owns camera traffic; progress checks read saved collection metadata. Every completed
image remains in the journal even if the session fails later. A failure is recorded and re-raised
after cleanup and USB closure. Do not rerun this function as an automatic retry.

In [ ]:
async def run_pcr(
  instrument: ABI7500FastDx,
  run_id: str,
  program: ThermalProgram,
  plan: CollectionPlan,
  expected_camera: str,
  output: Path,
  timeout: float,
) -> dict:
  """Execute one prepared optical method, preserving failure and cleanup details."""
  if timeout <= 0 or not math.isfinite(timeout):
    raise ValueError("Observation timeout must be finite and positive")
  plan.points(program)
  output.mkdir(parents=True, exist_ok=False)
  report = {
    "run_id": run_id,
    "program_hex": program.encode().hex(),
    "collection_plan": plan.to_dict(),
    "completed": False,
  }
  task = None
  cleanup_needed = False
  start_attempted = False
  try:
    await instrument.setup()
    identity = await instrument.get_identity()
    if identity.camera_description != expected_camera or identity.thermal_version != "G2.10":
      raise RuntimeError("Unexpected instrument identity")
    status = await instrument.get_status()
    optics = await instrument.get_optical_state()
    if (
      status.phase != "inactive"
      or status.state_code not in (1, 2)
      or max(status.sample_temperature, status.block_temperature) > 40
      or status.cover_temperature > 110
      or optics.to_dict()
      not in (
        {"filter": "0000", "motor": "0000", "lamp": "00000002"},
        {"filter": "0000", "motor": "0000", "lamp": "00010000"},
      )
    ):
      raise RuntimeError("Initial thermal or optical state differs; no actuation")
    report["identity"] = identity.camera_description
    report["stored_exposure_payload_hex"] = (
      await instrument.get_stored_exposure_times()
    ).raw_payload.hex()
    cleanup_needed = True
    await instrument.set_lamp(True, power=1)
    await asyncio.sleep(30)
    start_attempted = True
    await instrument.start_run(run_id, program, collection_plan=plan)
    task = asyncio.create_task(instrument.collect_fluorescence(run_id, timeout=timeout))
    while not task.done():
      await asyncio.wait({task}, timeout=5)
      records = await instrument.get_collection_records(run_id)
      print(
        f"Complete collections: {sum(r.state == 'complete' for r in records)}"
        f"/{len(plan.points(program))}"
      )
    collections = await task
    run = await instrument.get_run(run_id)
    if run.state != "completed" or len(collections) != len(plan.points(program)):
      raise RuntimeError("Run or collection count is incomplete")
    report["frame_count"] = sum(len(c.acquisitions) for c in collections)
    report["completed"] = True
  except BaseException as exc:
    report["error"] = repr(exc)
    raise
  finally:
    if task is not None and not task.done():
      task.cancel()
    if task is not None:
      try:
        await task
      except BaseException as exc:
        report["collector_error"] = repr(exc)
    try:
      if cleanup_needed:
        report["cleanup"] = await finish_pcr(instrument, run_id, expected_camera, start_attempted)
    except BaseException as exc:
      report["cleanup_error"] = repr(exc)
      raise
    finally:
      try:
        await instrument.stop()
        report["usb_closed"] = True
      finally:
        (output / "session.json").write_text(json.dumps(report, indent=2) + "\n")
  return report

## Load the plate and execute this approved session

Use a compatible, sealed 20 µL plate. The holder must be installed, the drawer closed and hands
clear before continuing. Keep the instrument connected exclusively to this Ubuntu host. Verify
that `usb_serial`, `expected_camera`, `journal_path`, `run_id` and the complete method above match
the approved session. Set `approved_session = True` only for that session.

The initial example uses water. Do not infer amplification or calibrated well values from it.
If a session fails, inspect `session.json` and the durable run before further hardware actions.

In [ ]:
if not approved_session:
  raise RuntimeError("Hardware execution is disabled until this session is explicitly approved")
if not usb_serial or not expected_camera or not journal_path.is_file():
  raise ValueError("Select this instrument's serial, identity and existing journal first")

instrument = ABI7500FastDx(serial_number=usb_serial, command_timeout=5, run_journal=journal_path)
report = await run_pcr(
  instrument, run_id, program, collection_plan, expected_camera, output, observation_timeout
)
print(report)

## Save original images — offline

Run this after USB has closed, including after a failed session whose run was reserved. It reads
the same journal without opening USB. Partial transfers are retained with their acquisition
state; they are never presented as complete frames. Existing exports are not overwritten.
The ZIP can be shared without moving the live journal. These files contain camera images and
metadata, not calibrated named-well values.

In [ ]:
def save_raw_collections(collections, destination: Path) -> Path:
  """Preserve each original transfer, reconstructed crop and collection association."""
  import zipfile

  hashes = {}
  with zipfile.ZipFile(destination, "x", zipfile.ZIP_DEFLATED) as archive:

    def add(name: str, data: bytes) -> None:
      archive.writestr(name, data)
      hashes[name] = hashlib.sha256(data).hexdigest()

    for collection in collections:
      folder = collection.record.collection_id
      add(f"{folder}/collection.json", json.dumps(collection.record.to_dict(), indent=2).encode())
      for acquisition in collection.acquisitions:
        prefix = f"{folder}/{acquisition.acquisition_id}"
        add(f"{prefix}/acquisition.json", json.dumps(acquisition.to_dict(), indent=2).encode())
        add(f"{prefix}/camera-original.bin", acquisition.wire_data)
        if acquisition.frame is not None:
          add(f"{prefix}/camera-image-u16le.bin", acquisition.frame.image_bytes)
    add("session.json", (destination.parent / "session.json").read_bytes())
    archive.writestr("sha256.json", json.dumps(hashes, indent=2))
  with zipfile.ZipFile(destination) as archive:
    for name, digest in hashes.items():
      if hashlib.sha256(archive.read(name)).hexdigest() != digest:
        raise RuntimeError(f"Export verification failed: {name}")
  return destination


saved = ABI7500FastDx(serial_number=usb_serial, run_journal=journal_path)
try:
  collections = await saved.get_collections(run_id)
  print(save_raw_collections(collections, output / "original-images.zip"))
finally:
  await saved.stop()

## Export raw per-well measurements — offline

If matching vendor masks are unavailable, supply an explicit `DiagnosticPlateRegions` JSON map
for this camera and physical filter. These regions remain provisional. They yield raw pixel
counts, with clipping and label-evidence flags, without background correction or calibration.
The export preserves all exposures and original frames. Never substitute another camera's map.
See {ref}`raw measurement limits <abi7500-raw-regions>`.

This cell uses the saved collections from the preceding cell and the completed checkpoint in
`session.json`; it does not query the instrument. It stays disabled until a region-map path is
supplied. No additional plate or hardware run is needed to process existing data.

In [ ]:
from pylabrobot.thermo_fisher.abi7500_fast_dx import (
  DiagnosticPlateRegions, ThermalRun, write_raw_run_archive,
)

regions_path = None  # Path to explicit diagnostic region JSON for this camera/filter.
raw_archive_path = None
if regions_path is not None:
  regions = DiagnosticPlateRegions.from_dict(json.loads(Path(regions_path).read_text()))
  session = json.loads((output / "session.json").read_text())
  checkpoint = ThermalRun.from_dict(session["cleanup"]["run"])
  raw_archive_path = write_raw_run_archive(
    output / "raw-well-measurements.zip", checkpoint, collections, (regions,)
  )
  print(raw_archive_path)

## Plot one fixed-exposure trace — offline

Choose a stage, step, physical filter, exposure and well explicitly. Do not combine exposure
settings or call a dye-only trace an amplification curve. Raw counts are not calibrated
fluorescence, Rn or Ct. Red markers retain clipped-region observations rather than dropping them.
This optional plotting cell requires Matplotlib and runs only after the raw export above.

In [ ]:
if raw_archive_path is not None:
  import csv
  import io
  import zipfile
  import matplotlib.pyplot as plt

  with zipfile.ZipFile(raw_archive_path) as archive:
    rows = list(csv.DictReader(io.StringIO(archive.read("raw-measurements.csv").decode())))
  stage, step, physical_filter, exposure, well = 2, 1, 5, 0.030, "A1"
  trace = sorted(
    (row for row in rows if int(row["stage"]) == stage and int(row["step_index"]) == step
     and int(row["physical_filter"]) == physical_filter and float(row["exposure"]) == exposure
     and row["label"] == well), key=lambda row: int(row["cycle"])
  )
  if not trace:
    raise ValueError("No saved measurements match the selected trace")
  fig, ax = plt.subplots()
  ax.plot([int(row["cycle"]) for row in trace], [float(row["mean"]) for row in trace])
  clipped = [row for row in trace if int(row["clipped_pixel_count"]) > 0]
  if clipped:
    ax.scatter([int(row["cycle"]) for row in clipped], [float(row["mean"]) for row in clipped],
               color="red", marker="x", label="Clipped region")
    ax.legend()
  ax.set(xlabel="Cycle", ylabel="Raw region mean (camera counts)",
         title=f"{well}, filter {physical_filter}, {exposure:g} s — provisional regions")
  plt.show()

## Export per-well fluorescence when unit-specific inputs are available

For a completed run, `export_run()` adds raw and corrected well/cycle/filter measurements and
exposure/saturation details. Supply this instrument's native background, uniformity and physical
filter masks. An older instrument's calibration must not be substituted. Establish the physical
well orientation before labeling results A1–H12. Keep files from the acquisition's calibration
set together. This processing neither renews calibration nor performs dye unmixing, Ct analysis
or assay interpretation. See [measurement processing](optics-and-measurements.md).

The following cell is disabled until all three paths are explicitly supplied.

In [ ]:
from pylabrobot.thermo_fisher.abi7500_fast_dx import (
  BackgroundCalibration,
  PlatePixelMask,
  UniformityCalibration,
)

background_path = None  # Native Background.csv from this instrument.
uniformity_path = None  # Matching uniformity.ini.
filter_5_mask_path = None  # Matching physical filter-5 ROI mask.
if all(path is not None for path in (background_path, uniformity_path, filter_5_mask_path)):
  saved = ABI7500FastDx(serial_number=usb_serial, run_journal=journal_path)
  try:
    archive = await saved.export_run(
      run_id,
      output / "well-measurements.zip",
      background=BackgroundCalibration.from_file(background_path),
      uniformity=UniformityCalibration.from_file(uniformity_path),
      masks=(PlatePixelMask.from_file(filter_5_mask_path, filter=5),),
    )
    print(archive)
  finally:
    await saved.stop()